## Machine learning pipeline — walkthrough

**Docs:** [`docs/architecture.md`](../docs/architecture.md) (modules, data flow), [`docs/ml_guide.md`](../docs/ml_guide.md) (story + example), [`README.md`](../README.md) (setup, CLI).

### Prerequisites

- **Working directory:** start Jupyter from the **repo root** or from `notebooks/` so the first code cell can set `ROOT` and `sys.path` correctly.
- **Dependencies:** `pip install -r requirements.txt` and `python -m spacy download en_core_web_sm` (spaCy model used in Step 3).
- **LLM (optional):** put `OPENAI_API_KEY` in a `.env` file at the repo root **before Step 9** if you want live API summaries and campaign copy; without it, the step still runs but falls back where the code path expects an API.
- **First run:** Step 1 may download the Kaggle trending CSV if `data/raw/` does not already have it (needs network).

In [1]:
import os
import sys
from pathlib import Path

_here = Path.cwd().resolve()
if (_here / "src").is_dir():
    ROOT = _here
elif (_here.parent / "src").is_dir():
    ROOT = _here.parent
else:
    raise RuntimeError("Start Jupyter from the repo root or from notebooks/.")

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Repository root:", ROOT)

Repository root: /Users/s0s0rn1/Desktop/Intuit/Mailchimp-Email-Marketing


In [2]:
from dotenv import load_dotenv

from IPython.display import display

import pandas as pd

from src.config.settings import Settings
from src.constants.pipeline_io import (
    TOPIC_INSIGHTS_FILENAME,
    VIDEOS_TEXT_BEFORE_TOPICS_FILENAME,
    VIDEOS_WITH_TOPICS_FILENAME,
)
from src.pipeline.pipeline_run import (
    TrendPipelineContext,
    _step_assign_topics,
    _step_enrich_documents,
    _step_load_dataset,
    _step_prepare_documents,
    _step_save_final_artifacts,
    _step_save_text_prep_checkpoint,
    _step_attach_topic_keywords,
    _step_marketer_insights,
    _step_offline_ranking_evaluation,
    _step_trend_scoring,
)
from src.pipeline.trend_engine import TrendPipelineEngine

load_dotenv(ROOT / ".env")
pd.set_option("display.max_colwidth", 72)

### Configuration and engine

**What:** `Settings` holds paths, `recent_trending_days` (default **7** calendar days of `trending_date`, newest anchor in the CSV), `max_rows`, `recency_half_life_hours` (publish→trending decay inside `TrendScorer`), model names, `llm_top_n`, `log_ranking_evaluation`, etc. `TrendPipelineEngine` wires loaders, NLP, BERTopic, scoring, evaluation, and the insight client—same as `main.py`.

**Why `max_rows` here:** after the date window, the loader keeps the **newest** rows first, then caps count—smaller caps make the notebook faster for teaching. Set `recent_trending_days=None` to use the full CSV history.


In [3]:
settings = Settings(max_rows=800, recent_trending_days=7)
engine = TrendPipelineEngine(settings)

print("recent_trending_days:", settings.recent_trending_days)
print("max_rows:", settings.max_rows)
print("recency_half_life_hours:", settings.recency_half_life_hours)
print("processed_data_dir:", settings.processed_data_dir)
print("output_dir:", settings.output_dir)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


recent_trending_days: 7
max_rows: 800
recency_half_life_hours: 96.0
processed_data_dir: data/processed
output_dir: outputs


### Pipeline context

**What:** `TrendPipelineContext` is a small bag: `engine`, `videos_df` (one row per video, growing columns as steps run), and `topic_insights_df` (one row per topic, filled from Step 6 onward).

**Why:** each `_step_*` mutates `ctx` in place—same pattern as `run_trend_pipeline()` in code. **Run steps in order**; skipping or reordering breaks downstream assumptions.


In [4]:
ctx = TrendPipelineContext(engine=engine)

#### Step 1: Load dataset

**What:** Bring trending video rows into a pandas `DataFrame`.

**How:** `TrendingDatasetLoader` reads the regional CSV (default: Kaggle `USvideos.csv`), keeps schema columns, optionally restricts to the last **`Settings.recent_trending_days`** of `trending_date`, sorts newest-first, applies **`max_rows`**, then `validate_trending_video_rows` checks types/required fields.

**Why:** Everything downstream assumes clean, validated tabular input; no embeddings or topics yet.

The preview lists key columns later used in scoring and enrichment: `views`, `likes`, `dislikes`, `comment_count`, `trending_date`, and `publish_time` feed **`TrendScorer.score`** in Step 6 (with `topic` from Step 5). `description` appears only if enabled in `Settings`.


In [5]:
_step_load_dataset(ctx)

v = ctx.videos_df
assert v is not None
print("videos_df shape:", v.shape)
cols = [
    c
    for c in (
        "title",
        "tags",
        "views",
        "likes",
        "dislikes",
        "comment_count",
        "trending_date",
        "publish_time",
        "description",
    )
    if c in v.columns
]
display(v[cols].head(2))


Loading dataset from: data/raw/USvideos.csv
recent_trending_days=7: kept 1400 rows (trending_date 2018-06-08 … 2018-06-14, was 40949).
max_rows=800: using 800 newest-by-trending_date rows (was 1400).
videos_df shape: (800, 8)


,title,tags,views,likes,dislikes,comment_count,trending_date,publish_time
40948,Official Call of Duty®: Black Ops 4 — Multiplayer Reveal Trailer,"call of duty|""cod""|""activision""|""Black Ops 4""",10306119,357079,212976,144795,18.14.06,2018-05-17T17:09:38.000Z
40811,Why 350°F is the magic number for baking,"baking|""maillard reaction""|""s pen""|""Vox.com""|""vox""|""explain""|""explai...",799411,16092,652,1551,18.14.06,2018-06-07T12:00:02.000Z


#### Step 2: Build documents

**What:** One text field per video for NLP.

**How:** Concatenate **title**, **tags**, and optionally **description** (`Settings.use_description`) into a column `document`.

**Why:** Clustering and embeddings need a single string per row; this matches how `main.py` builds the document.


In [6]:
_step_prepare_documents(ctx)

v = ctx.videos_df
cols = [c for c in ("title", "document") if c in v.columns]
display(v[cols].head(2))

,title,document
40948,Official Call of Duty®: Black Ops 4 — Multiplayer Reveal Trailer,Official Call of Duty®: Black Ops 4 — Multiplayer Reveal Trailer cal...
40811,Why 350°F is the magic number for baking,"Why 350°F is the magic number for baking baking ""maillard reaction"" ..."


#### Step 3: spaCy normalization (lemmatized text)

**What:** Turn raw `document` into lighter, more comparable text.

**How:** `SpacyPreprocessor` lemmatizes, drops stopwords/noise, and writes `cleaned_text`.

**Why:** Reduces spelling/noise so sentence embeddings and BERTopic’s bag-of-words statistics are more stable; **embeddings use `cleaned_text`**, not the original title alone.


In [7]:
_step_enrich_documents(ctx)

v = ctx.videos_df
cols = [c for c in ("title", "document", "cleaned_text") if c in v.columns]
display(v[cols].head(2))

,title,document,cleaned_text
40948,Official Call of Duty®: Black Ops 4 — Multiplayer Reveal Trailer,Official Call of Duty®: Black Ops 4 — Multiplayer Reveal Trailer cal...,official duty black ops 4 multiplayer reveal trailer duty cod activi...
40811,Why 350°F is the magic number for baking,"Why 350°F is the magic number for baking baking ""maillard reaction"" ...",350 f magic number bake bake maillard reaction s pen vox explain exp...


#### Step 4: Save text-prep checkpoint (before topics)

**What:** Persist the video table **after** text prep, **before** expensive embedding/topic work.

**How:** Writes `videos_text_before_topics.csv` under `processed_data_dir`.

**Why:** Lets you **reuse** `document` / `cleaned_text` without re-running spaCy, and makes runs **auditable** (diff/checkpoint).


In [8]:
_step_save_text_prep_checkpoint(ctx)

ck = Path(settings.processed_data_dir) / VIDEOS_TEXT_BEFORE_TOPICS_FILENAME
print("Checkpoint:", ck.resolve(), "exists:", ck.is_file())

Checkpoint: /Users/s0s0rn1/Desktop/Intuit/Mailchimp-Email-Marketing/data/processed/videos_text_before_topics.csv exists: True


#### Step 5: Embeddings + topic assignment

**What:** Assign each video to a **topic cluster** (integer id) and a confidence.

**How:** `SentenceTransformer` encodes `cleaned_text` into vectors; **BERTopic** (`TopicModeler.fit_transform`) clusters in embedding space and fits topic representations. Rows get `topic` and `topic_confidence`. **−1** = outlier/noise bucket (excluded from topic-level scoring later).

**Why:** Groups “similar” videos for aggregate trends; **clustering follows embeddings**; keyword statistics inside BERTopic still use the text side of the pipeline.


In [9]:
_step_assign_topics(ctx)

v = ctx.videos_df
cols = [c for c in ("title", "topic", "topic_confidence") if c in v.columns]
display(v[cols].head(5))

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

2026-04-27 16:43:15,068 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-27 16:43:20,542 - BERTopic - Dimensionality - Completed ✓
2026-04-27 16:43:20,542 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-27 16:43:20,575 - BERTopic - Cluster - Completed ✓
2026-04-27 16:43:20,576 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-27 16:43:20,601 - BERTopic - Representation - Completed ✓


,title,topic,topic_confidence
40948,Official Call of Duty®: Black Ops 4 — Multiplayer Reveal Trailer,17,0.952714
40811,Why 350°F is the magic number for baking,1,0.791826
40821,“Bonding” with Grandma,-1,0.002277
40820,Knife Expert Guesses Cheap vs. Expensive Knives | Price Points | Epi...,1,0.410593
40819,The Internet - Come Over (Official Video),7,1.000000


#### Step 6: Trend scoring

**What:** One row per **topic** with final LambdaMART-based ranking outputs used by the dashboard/LLM flow.

**How:** `TrendScorer.score` computes topic features, assigns each topic to a ranking segment, trains LambdaMART on **(`date`, `ranking_segment`)** query groups using segment-local labels, then returns final topic-level ranking fields (including `ranking_segment`, `segment_rank`, and `trend_score`).

**Why:** This table is the **final scored output** (for serving and downstream steps). Training uses `(date, ranking_segment)` query groups inside `TrendScorer.score`; intermediate label rows and pairwise construction are not part of the public surface.



In [10]:
_step_trend_scoring(ctx)

t = ctx.topic_insights_df
assert t is not None
cols = [
    "topic",
    "ranking_segment",
    "segment_rank",
    "trend_score",
    "volume",
    "momentum",
    "avg_proxy_ctr_recency",
    "avg_views",
    "avg_likes",
]
cols = [c for c in cols if c in t.columns]
display(t[cols].head(5))

,topic,ranking_segment,segment_rank,trend_score,volume,momentum,avg_proxy_ctr_recency,avg_views,avg_likes
0,4,food,1.0,1.000000,40,0.076923,0.013841,3.348775e+06,92174.000000
1,4,technology,1.0,0.991350,40,0.076923,0.013841,3.348775e+06,92174.000000
2,4,general,1.0,0.965839,40,0.076923,0.013841,3.348775e+06,92174.000000
3,2,beauty_lifestyle,1.0,0.957682,45,-0.153846,0.018212,2.099249e+06,99229.244444
4,5,entertainment,1.0,0.945241,35,0.000000,0.012227,1.353877e+07,336421.285714


#### Step 7: Offline ranking evaluation

**What:** Optional **console-only** metric: proxy **NDCG@K** (no human labels).

**How:** Compares ordering by **`trend_score`** to an “ideal” ordering by blended gain on the top-**K** rows (K = `llm_top_n`, capped by table length). Blended gain is `0.5*ctr_recency_norm + 0.3*volume_norm + 0.2*momentum_norm`, where `ctr_recency_norm` is based on `avg_proxy_ctr_recency`. Toggle with **`log_ranking_evaluation`** on `Settings`.

**Why:** Cheap **sanity check** that the ranker is aligned with engagement quality + scale + momentum together—not accuracy, not business KPIs. **How to read:** 1.0 = same order as sorting by blended gain on that slice; lower = more reordering.


In [11]:
_step_offline_ranking_evaluation(ctx)


Ranking evaluation (before LLM enrichment)
------------------------------------------------------------
Proxy NDCG (trend_score order vs ideal by blended gain)
  Interpretation: 1.0 = same ordering as sorting by blended gain; <1.0 = ranker reorders vs blended proxy.
  k: 10.0
  gain_formula: 0.5*ctr_recency_norm + 0.3*volume_norm + 0.2*momentum_norm
  score_col: trend_score
  dcg@k (score order): 3.665293388354445
  idcg@k (ideal by gain): 4.17708402758557
  ndcg@k (proxy): 0.8774765755605474
------------------------------------------------------------


#### Step 8: Attach topic keywords

**What:** Human-readable **labels and keyword lists** per topic from the **fitted** BERTopic model.

**How:** `add_topic_keyword_columns` fills `topic_keywords`, **`dominant_topic_keywords`**, and **`topic_label`** (short string from the first keywords).

**Why:** Step 9 needs lexical context for taxonomy/coherence checks, display names, and the LLM prompt; **dominant** keywords are what feed classification and the API text (see `enrich_topic_insights_marketer_fields`).


In [12]:
_step_attach_topic_keywords(ctx)

t = ctx.topic_insights_df
assert t is not None
cols = ["topic", "topic_label", "topic_keywords", "dominant_topic_keywords"]
cols = [c for c in cols if c in t.columns]
display(t[cols].head(5))

,topic,topic_label,topic_keywords,dominant_topic_keywords
0,4,"nintendo, let, halo, game, creed","[nintendo, let, halo, game, creed, switch, eevee, pikachu]","[nintendo, let, halo, game, creed]"
1,4,"nintendo, let, halo, game, creed","[nintendo, let, halo, game, creed, switch, eevee, pikachu]","[nintendo, let, halo, game, creed]"
2,4,"nintendo, let, halo, game, creed","[nintendo, let, halo, game, creed, switch, eevee, pikachu]","[nintendo, let, halo, game, creed]"
3,2,"diy, room, shibori, decor, dye","[diy, room, shibori, decor, dye, vogue, master, tour]","[diy, room, shibori, decor, dye]"
4,5,"daddy, yankee, party, welcome, music","[daddy, yankee, party, welcome, music, diplo, balloon, gorillaz]","[daddy, yankee, party, welcome, music]"


#### Step 9: Marketer insights

**What:** Enrich each topic with **taxonomy**, **coherence**, sample titles, optional **LLM** JSON (`summary`, `campaign_copy`, …) for the first **`llm_top_n`** rows (by dataframe index).

**How:** Rule-based checks (`topic_is_coherent`, `classify_trend_type`, `TopicNamer`); if coherent and safe path, **`InsightGenerator.generate_insight`** calls OpenAI with `TREND_INSIGHT_PROMPT_TEMPLATE`; otherwise canned copy.

**Why:** Turns clusters into **marketer-facing** text; the code cell’s **truncated prompt** preview matches `InsightGenerator` for the top **`trend_score`** row (API may still skip incoherent rows).


In [13]:
_step_marketer_insights(ctx)

from src.constants.llm_prompts import TREND_INSIGHT_PROMPT_TEMPLATE

t = ctx.topic_insights_df
assert t is not None
print("topic_insights_df shape:", t.shape)
cols = [
    "topic",
    "topic_label",
    "ranker",
    "trend_score",
    "volume",
    "momentum",
    "avg_views",
    "avg_likes",
    "avg_proxy_ctr_recency",
    "marketing_safe",
]
cols = [c for c in cols if c in t.columns]
display(t[cols].head(8))

row0 = t.iloc[0]
llm_input = TREND_INSIGHT_PROMPT_TEMPLATE.format(
    topic_keywords=row0["dominant_topic_keywords"],
    sample_titles=row0["sample_titles"],
    trend_type=row0["trend_type"],
    avg_views=int(row0["avg_views"]),
    avg_likes=int(row0["avg_likes"]),
    momentum=f"{row0['momentum']:.2f}",
    avg_proxy_ctr_recency=f"{float(row0.get('avg_proxy_ctr_recency', 0.0)):.4f}",
)
print()
print("Sample LLM input (top `trend_score` row; truncated)")
print("=" * 60)
_max = 2400
print(llm_input[:_max] + ("" if len(llm_input) <= _max else "\n…"))
if len(llm_input) > _max:
    print(f"(truncated; full string is {len(llm_input)} chars — see TREND_INSIGHT_PROMPT_TEMPLATE in src/constants/llm_prompts.py)")

topic_insights_df shape: (49, 37)


,topic,topic_label,trend_score,volume,momentum,avg_views,avg_likes,avg_proxy_ctr_recency,marketing_safe
0,4,"nintendo, let, halo, game, creed",1.000000,40,0.076923,3.348775e+06,92174.000000,0.013841,True
1,4,"nintendo, let, halo, game, creed",0.991350,40,0.076923,3.348775e+06,92174.000000,0.013841,True
2,4,"nintendo, let, halo, game, creed",0.965839,40,0.076923,3.348775e+06,92174.000000,0.013841,True
3,2,"diy, room, shibori, decor, dye",0.957682,45,-0.153846,2.099249e+06,99229.244444,0.018212,False
4,5,"daddy, yankee, party, welcome, music",0.945241,35,0.000000,1.353877e+07,336421.285714,0.012227,False
5,2,"diy, room, shibori, decor, dye",0.922063,45,-0.153846,2.099249e+06,99229.244444,0.018212,False
6,1,"knife, taste, bake, good, skittle",0.918813,53,0.071429,2.177392e+06,61696.226415,0.011119,True
7,11,"fnaf, box, fortnite, fort, game",0.895614,29,0.125000,3.291897e+06,82476.241379,0.010846,True



Sample LLM input (top `trend_score` row; truncated)

You generate grounded marketing insights for a Mailchimp-style trend engine.

Return ONLY valid JSON with these keys:
- summary
- campaign_angle
- suggested_subject
- email_hook
- marketing_safe

Inputs:
- topic_keywords: ['nintendo', 'let', 'halo', 'game', 'creed']
- sample_titles: ['Apartment Atrocities', "Assassin's Creed Odyssey: E3 2018 Official World Premiere Trailer | Ubisoft [NA]", 'YouTube Live at E3 2018: Monday with Ninja, Marshmello, PlayStation, Ubisoft, Todd Howard']
- trend_type: technology
- avg_views: 3348775
- avg_likes: 92174
- momentum: 0.08
- avg_proxy_ctr_recency: 0.0138

Rules:
1. Use only the provided inputs.
2. If a claim cannot be directly supported by topic_keywords or sample_titles, do not include it.
3. Do not invent specific shows, products, events, controversies, relationships, or storylines.
4. Treat names as context, not the main theme unless clearly dominant across multiple keywords or multiple samp

#### Step 10: Save final outputs

**What:** Persist the **final** video-level and topic-level tables for the app and reuse.

**How:** `save_final_artifacts` validates rows and writes **`videos_with_topics.csv`** and **`topic_insights.csv`** under `output_dir` (paths in `src/constants/pipeline_io.py`).

**Why:** **`streamlit run dashboard.py`** and `python -m src.evaluation` expect these paths; reproducible handoff from batch run to UI.


In [14]:
_step_save_final_artifacts(ctx)

out = Path(settings.output_dir)
vwt = out / VIDEOS_WITH_TOPICS_FILENAME
tic = out / TOPIC_INSIGHTS_FILENAME
print("videos_with_topics:", vwt.resolve(), vwt.is_file())
print("topic_insights:", tic.resolve(), tic.is_file())

videos_with_topics_df = ctx.videos_df
topic_insights_df = ctx.topic_insights_df

Saved final outputs to: /Users/s0s0rn1/Desktop/Intuit/Mailchimp-Email-Marketing/outputs
videos_with_topics: /Users/s0s0rn1/Desktop/Intuit/Mailchimp-Email-Marketing/outputs/videos_with_topics.csv True
topic_insights: /Users/s0s0rn1/Desktop/Intuit/Mailchimp-Email-Marketing/outputs/topic_insights.csv True


### Final output — top trends (pretty-print)

**What:** Same **pretty-print** loop as `main.py`: walk **`topic_insights_df`** in **`trend_score`** order and print scores plus **`summary`** / **`campaign_copy`**.

**Why:** Quick **human-readable** demo of the end product without opening Streamlit. Does not re-run the pipeline.


In [15]:
assert videos_with_topics_df is not None and topic_insights_df is not None

top_n = settings.top_n_topics_to_show

for _, row in topic_insights_df.head(top_n).iterrows():
    suggestion = row["campaign_copy"]
    print("\n" + "=" * 60)
    print(f"Trend: {row['topic_label']}")
    print(f"Trend Score: {row['trend_score']:.2f}")
    print(f"Volume: {int(row['volume'])}")
    print(f"Avg Views: {int(row['avg_views']):,}")
    print(f"Avg Likes: {int(row['avg_likes']):,}")
    print(f"Momentum: {row['momentum']:.2f}")
    print("\nSummary:")
    print(row["summary"])
    print("\nCampaign copy:")
    print(f"  Suggested Subject: {suggestion['suggested_subject']}")
    print(f"  Campaign Angle: {suggestion['campaign_angle']}")
    print(f"  Email Hook: {suggestion['email_hook']}")


Trend: nintendo, let, halo, game, creed
Trend Score: 1.00
Volume: 40
Avg Views: 3,348,775
Avg Likes: 92,174
Momentum: 0.08

Summary:
Content related to popular gaming franchises and technology, supported by keywords like Nintendo, Halo, and Assassin's Creed, shows steady interest with moderate engagement levels. This cluster reflects ongoing enthusiasm for gaming content around well-known titles without sharp momentum shifts.

Campaign copy:
  Suggested Subject: Explore Classic Hits in Gaming Today
  Campaign Angle: Leverage steady interest in popular gaming franchises by highlighting iconic game features and timeless gameplay experiences in marketing campaigns.
  Email Hook: Dive into the timeless worlds of Nintendo, Halo, and Assassin's Creed—where gaming legends continue to captivate players worldwide.

Trend: nintendo, let, halo, game, creed
Trend Score: 0.99
Volume: 40
Avg Views: 3,348,775
Avg Likes: 92,174
Momentum: 0.08

Summary:
Content around popular gaming franchises such as